# 01 — Búsqueda G1: Proyecto Oráculo

**Integrante:** Johan Stiven Osorio Grisales  
**Grupo:** G1 · **Semana:** 2  
**Modelo:** Qwen/Qwen3-1.7B · **Temperatura:** 0.0

## Propósito

Documentar la búsqueda de una configuración de prompt que maximice la proporción de instancias que satisfacen todas las restricciones del oráculo.

La ejecución base utiliza búsqueda por coordenadas con screening de 10 instancias, refinamiento de dos finalistas por ranura sobre las 100 instancias de búsqueda y una ronda.

## Resultado registrado

- Configuración mejor encontrada: `rol=1, estrategia=5, formato=1, estilo=3, cierre=6, temperatura=0.0`.
- Precisión en búsqueda: 9/100 = **9,0 %**.
- Precisión en validación: 8/87 = **9,2 %**.
- Historial: 46 evaluaciones; cinco coordenadas completadas.
- No hubo mejoras aceptadas sobre el baseline.

**Alcance:** es la mejor configuración observada por esta ejecución; no se demuestra optimalidad global. La demostración de NP-dureza desde 3-SAT se documenta por separado y permanece pendiente de una reducción formal verificada.

## 1. Preparar el entorno de Colab
Ejecutar con entorno de ejecución GPU (Tesla T4).

In [ ]:
import os, sys, subprocess, json, random
from pathlib import Path

REPOSITORIO = "https://github.com/StivenLH24/ADA-2026-2-GP01.git"
RUTA_PROYECTO = Path("/content/ADA-2026-2-GP01")
RAMA = "trabajo-g1"

if not (RUTA_PROYECTO / ".git").exists():
    if RUTA_PROYECTO.exists():
        import shutil
        shutil.rmtree(RUTA_PROYECTO)
    subprocess.run([
        "git", "clone", "--branch", RAMA, "--single-branch",
        REPOSITORIO, str(RUTA_PROYECTO)
    ], check=True)
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", RAMA],
        cwd=RUTA_PROYECTO, check=True
    )

os.chdir(RUTA_PROYECTO)
if str(RUTA_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RUTA_PROYECTO))

print("Proyecto:", os.getcwd())
print("Rama:", RAMA)
!git status --short --branch

## 2. Dependencias y verificadores
La instalación reproduce las dependencias utilizadas en la ejecución documentada.

In [ ]:
%pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict

!python -m spacy download en_core_web_sm -q

RUTA_VERIFICADORES = "/content/open-instruct"
if not os.path.isdir(os.path.join(RUTA_VERIFICADORES, ".git")):
    !git clone -q https://github.com/allenai/open-instruct /content/open-instruct

import sys
if RUTA_VERIFICADORES not in sys.path:
    sys.path.insert(0, RUTA_VERIFICADORES)

print("Dependencias y verificadores preparados.")

In [ ]:
import io, urllib.request, zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]

for carpeta, nombre in PAQUETES:
    destino = f"/root/nltk_data/{carpeta}"
    os.makedirs(destino, exist_ok=True)
    url = f"{NLTK_DATA}/{carpeta}/{nombre}.zip"
    with urllib.request.urlopen(url) as respuesta:
        contenido = respuesta.read()
    with zipfile.ZipFile(io.BytesIO(contenido)) as archivo:
        archivo.extractall(destino)

print("Recursos NLTK preparados.")

## 3. Datos, GPU, caché y modelo
La caché persistente evita repetir generaciones ya realizadas. No la borres entre sesiones.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CARPETA_DATOS = Path("/content/drive/MyDrive/ADA_G1")
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
CACHE_ORACULO = str(CARPETA_DATOS / "cache_oraculo_G1.json")
HISTORIAL_BUSQUEDA = CARPETA_DATOS / "historial_coordinate_search_G1.json"

import torch
from open_instruct.IFEvalG import instructions_registry
from ayudas import cargar_datos, dividir, cargar_modelo
from oraculo import Oraculo, CATALOGO, RANURAS, LIMPIA
from src.optimizador import CoordinateSearch
from src.reanudador import reanudar_busqueda_g1

assert torch.cuda.is_available(), "Seleccionar GPU T4 en Colab."
print("GPU:", torch.cuda.get_device_name(0))

datos = cargar_datos()
busqueda, validacion = dividir(datos)
assert len(busqueda) == 100 and len(validacion) == 87

modelo = cargar_modelo("qwen17b")
oraculo = Oraculo(
    modelo=modelo,
    datos=busqueda,
    validacion=validacion,
    cache=CACHE_ORACULO,
    max_new_tokens=384,
    lote=1
)

print("Respuestas en caché:", len(oraculo.cache))

## 4. Cargar la configuración de entrega
El notebook y el archivo JSON deben contener exactamente la misma configuración.

In [ ]:
with open("entrega.json", encoding="utf-8") as f:
    entrega = json.load(f)

assert entrega["grupo"] == "G1"
assert entrega["semana"] == 2

CONFIG_GANADORA = entrega["config"]
assert set(CONFIG_GANADORA) == set(RANURAS) | {"temperatura"}
assert CONFIG_GANADORA["temperatura"] == 0.0

print(json.dumps(entrega, indent=2, ensure_ascii=False))

## 5. Leer el historial experimental
La tabla se construye desde el JSON persistido en Drive, no desde una transcripción manual de la consola.

In [ ]:
import pandas as pd
from collections import Counter

assert HISTORIAL_BUSQUEDA.exists(), (
    "No se encontró el historial. Copiar historial_coordinate_search_G1.json "
    "a la carpeta MyDrive/ADA_G1."
)

with open(HISTORIAL_BUSQUEDA, encoding="utf-8") as f:
    historial = json.load(f)

evaluaciones = historial["evaluaciones"]
resumen_coordenadas = historial["resumen_coordenadas"]

tabla_evaluaciones = pd.DataFrame([{
    "evaluacion": e["evaluacion"],
    "etapa": e["etapa"],
    "ranura": e.get("ranura"),
    "indice": e.get("indice_candidato"),
    "precision_pct": e["precision_porcentaje"],
    "correctas": e["correctas"],
    "fallidas": e["fallidas"],
    "n": e["instancias"]
} for e in evaluaciones])

display(tabla_evaluaciones)
print("Total de evaluaciones:", len(evaluaciones))
print("Coordenadas completadas:", [x["ranura"] for x in resumen_coordenadas])
print("Configuración registrada:", historial["configuracion_actual"])
print("Precisión registrada:", historial["precision_actual"])

## 6. Resultados de refinamiento por coordenada

In [ ]:
refinamientos = [
    e for e in evaluaciones
    if e["etapa"] == "refinamiento" and e["instancias"] == 100
]

tabla_refinamientos = pd.DataFrame([{
    "ranura": e["ranura"],
    "indice_candidato": e["indice_candidato"],
    "correctas": e["correctas"],
    "fallidas": e["fallidas"],
    "precision_pct": e["precision_porcentaje"]
} for e in refinamientos])

display(tabla_refinamientos)

baseline = next(e for e in evaluaciones if e["etapa"] == "baseline")
print(f"Baseline: {baseline['correctas']}/{baseline['instancias']} = "
      f"{baseline['precision_porcentaje']:.1f}%")

## 7. Análisis de trazas de los refinamientos
Cada traza identifica la primera familia de restricción incumplida para una instancia. No representa necesariamente todos los fallos de esa respuesta.

In [ ]:
filas_fallos = []
for e in refinamientos:
    familias = Counter(t.get("violo") for t in e.get("trazas", []) if t.get("violo"))
    for familia, frecuencia in familias.items():
        filas_fallos.append({
            "ranura": e["ranura"],
            "indice": e["indice_candidato"],
            "familia": familia,
            "frecuencia": frecuencia
        })

tabla_fallos = pd.DataFrame(filas_fallos)
display(tabla_fallos.sort_values(["ranura", "frecuencia"], ascending=[True, False]))

## 8. Validación final con oraculo.validar
Esta llamada usa la partición de validación entregada a `Oraculo`. La prueba privada que realice el docente es independiente y no está incluida en los archivos del equipo.

In [ ]:
resultado_validacion = oraculo.validar(
    CONFIG_GANADORA,
    semilla=2026
)

correctas = resultado_validacion.n - len(resultado_validacion.trazas)
print("\nVALIDACIÓN FINAL")
print("Configuración:", CONFIG_GANADORA)
print("Precisión:", f"{resultado_validacion.precision:.2%}")
print("Instancias:", resultado_validacion.n)
print("Correctas:", correctas)
print("Fallidas:", len(resultado_validacion.trazas))

familias_validacion = Counter(
    t.get("violo") for t in resultado_validacion.trazas if t.get("violo")
)
display(pd.DataFrame(
    familias_validacion.most_common(),
    columns=["familia_primera_falla", "frecuencia"]
))

oraculo._guardar()

## 9. Visualizaciones

In [ ]:
import matplotlib.pyplot as plt

etiquetas = [
    f"{e['ranura']}={e['indice_candidato']}"
    for e in refinamientos
]
valores = [e["precision_porcentaje"] for e in refinamientos]

plt.figure(figsize=(12, 5))
plt.bar(etiquetas, valores)
plt.axhline(baseline["precision_porcentaje"], linestyle="--",
            label="Baseline 9,0 %")
plt.ylabel("Precisión sobre las 100 instancias (%)")
plt.xlabel("Candidato refinado")
plt.title("Coordinate Search G1 — refinamientos")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
nombres = [x[0] for x in familias_validacion.most_common(8)]
frecuencias = [x[1] for x in familias_validacion.most_common(8)]
plt.barh(nombres[::-1], frecuencias[::-1])
plt.xlabel("Frecuencia")
plt.title("Familias de primera falla — validación")
plt.tight_layout()
plt.show()

## 10. Conclusión

La búsqueda por coordenadas terminó una ronda con 46 evaluaciones registradas y cinco coordenadas completadas. Ningún cambio superó estrictamente el baseline de 9,0 %. La configuración del archivo `entrega.json` obtuvo 9,2 % en las 87 instancias de validación.

La evidencia corresponde a la mejor configuración encontrada por el método y presupuesto descritos. No constituye una garantía de óptimo global. La reducción desde 3-SAT requerida por la guía se desarrolla en una sección académica independiente y no se considera demostrada hasta verificar la transformación y la equivalencia en ambos sentidos.